In [7]:
import pandas as pd

experiments = ["eth_micron", 'moz_vam','nga_mics', 'zwe_mics','yem_mvam','lka_micron','lka_vam','nga_micron']
base_path = '/data/shared/fsibilla/clean_code/Q1/experiments/{}/full.csv'

In [8]:
results = []

for exp in experiments:
    path = base_path.format(exp)
    try:
        df = pd.read_csv(path, low_memory=False)
    except FileNotFoundError:
        print(f"NOT FOUND: {exp}")
        continue

    # detect psu column
    psu_col = 'psu' if 'psu' in df.columns else None
    if psu_col is None:
        print(f"{exp}: no 'psu' column found. Columns: {list(df.columns)}")
        continue

    # detect adm columns
    adm1_col = 'adm1name' if 'adm1name' in df.columns else None
    adm2_col = 'adm2name' if 'adm2name' in df.columns else None

    total_rows     = len(df)
    n_unique_global = df[psu_col].nunique()
    n_rows_per_psu  = total_rows / n_unique_global if n_unique_global > 0 else None

    # how many PSUs appear in more than one ADM1?
    psu_crosses_adm1 = None
    if adm1_col:
        psu_adm1_counts = df.groupby(psu_col)[adm1_col].nunique()
        psu_crosses_adm1 = (psu_adm1_counts > 1).sum()

    # how many PSUs appear in more than one ADM2?
    psu_crosses_adm2 = None
    if adm2_col:
        psu_adm2_counts = df.groupby(psu_col)[adm2_col].nunique()
        psu_crosses_adm2 = (psu_adm2_counts > 1).sum()

    # are PSUs unique within each ADM1?
    unique_within_adm1 = None
    if adm1_col:
        # count distinct PSUs per ADM1, sum them, compare to global unique
        n_unique_within_adm1 = df.groupby(adm1_col)[psu_col].nunique().sum()
        unique_within_adm1 = (n_unique_within_adm1 == n_unique_global)

    # are PSUs unique within each ADM2?
    unique_within_adm2 = None
    if adm2_col:
        n_unique_within_adm2 = df.groupby(adm2_col)[psu_col].nunique().sum()
        unique_within_adm2 = (n_unique_within_adm2 == n_unique_global)

    results.append({
        'experiment':         exp,
        'total_rows':         total_rows,
        'unique_PSUs_global': n_unique_global,
        'avg_rows_per_PSU':   round(n_rows_per_psu, 1) if n_rows_per_psu else None,
        'PSUs_cross_adm1':    psu_crosses_adm1,
        'PSUs_cross_adm2':    psu_crosses_adm2,
        'unique_within_adm1': unique_within_adm1,
        'unique_within_adm2': unique_within_adm2,
    })

summary = pd.DataFrame(results)
summary

yem_mvam: no 'psu' column found. Columns: ['id', 'survey_adm1name', 'survey_adm2name', 'adm1name', 'adm1code', 'adm2name', 'adm2code', 'log_exp_pp', 'rCSI', 'FCS', 'wscore_1', 'rfh_avg_2', 'r3q_2', 'vim_avg_2', 'rfh_avg_1', 'r3q_1', 'vim_avg_1', 'adm1geometry', 'adm2geometry', 'climate_code_merge_status', 'adm2_climate_source', 'entropy_2', 'entropy_1']


,experiment,total_rows,unique_PSUs_global,avg_rows_per_PSU,PSUs_cross_adm1,PSUs_cross_adm2,unique_within_adm1,unique_within_adm2
0,eth_micron,6214,498,12.5,0,0.0,True,True
1,moz_vam,11080,1059,10.5,0,0.0,True,True
2,nga_mics,39572,2079,19.0,0,NaN,True,None
3,zwe_mics,11043,462,23.9,0,NaN,True,None
4,lka_micron,19911,2411,8.3,0,0.0,True,True
5,lka_vam,14488,1452,10.0,66,92.0,False,False
6,nga_micron,22106,2154,10.3,0,NaN,True,None


In [9]:
# ============================================================
# OVERWRITE full.csv WITH ADM2-LEVEL UNIQUE PSU IDS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

FULL_CSV_PATH = Path("/data/shared/fsibilla/clean_code/Q1/experiments/lka_vam/full.csv")

df = pd.read_csv(FULL_CSV_PATH)

psu_col = "psu"
psu_raw_col = "psu_raw"
psu_unique_col = "psu_unique"

# Prefer ADM2 codes if available; fall back to ADM2 names if needed.
if {"adm1code", "adm2code", psu_col}.issubset(df.columns):
    psu_group_cols = ["adm1code", "adm2code", psu_col]
elif {"adm1name", "adm2name", psu_col}.issubset(df.columns):
    psu_group_cols = ["adm1name", "adm2name", psu_col]
else:
    raise ValueError(
        "Cannot create ADM2-level unique PSU. "
        "Need either ['adm1code', 'adm2code', 'psu'] "
        "or ['adm1name', 'adm2name', 'psu']."
    )

print("Using columns to make PSU unique:")
print(psu_group_cols)

# Keep original PSU only once.
if psu_raw_col not in df.columns:
    df[psu_raw_col] = df[psu_col]

for c in psu_group_cols:
    df[c] = df[c].astype(str).str.strip()

df[psu_unique_col] = (
    df[psu_group_cols[0]]
    + "||" + df[psu_group_cols[1]]
    + "||" + df[psu_col]
)

print("Rows:", len(df))
print("Raw PSU IDs:", df[psu_raw_col].nunique())
print("ADM2-level unique PSU IDs:", df[psu_unique_col].nunique())

# Replace psu with corrected unique PSU so downstream scripts can keep psu_col = "psu".
df[psu_col] = df[psu_unique_col]

# Overwrite full.csv
df.to_csv(FULL_CSV_PATH, index=False)

print("Overwritten corrected file:")
print(FULL_CSV_PATH)

# Quick check
check = (
    df.groupby(psu_col)
      .agg(
          n_adm1=("adm1name", "nunique") if "adm1name" in df.columns else (psu_col, "size"),
          n_adm2=("adm2name", "nunique") if "adm2name" in df.columns else (psu_col, "size"),
      )
      .reset_index()
)

if {"adm1name", "adm2name"}.issubset(df.columns):
    problematic = check[(check["n_adm1"] > 1) | (check["n_adm2"] > 1)]
    print("Corrected PSU IDs reused across ADM1/ADM2:", len(problematic))
    if len(problematic) > 0:
        display(problematic.head(20))

Using columns to make PSU unique:
['adm1code', 'adm2code', 'psu']
Rows: 14488
Raw PSU IDs: 1452
ADM2-level unique PSU IDs: 1557


Overwritten corrected file:
/data/shared/fsibilla/clean_code/Q1/experiments/lka_vam/full.csv
Corrected PSU IDs reused across ADM1/ADM2: 0
